## Análisis de Homicidios

In [2]:
import pandas as pd

In [10]:

df = pd.read_csv("../staging/processed/homicidios_clean.csv")


### Homicidios por año

In [11]:
def homicidios_por_anio(df: pd.DataFrame) -> pd.DataFrame:
    resultado = (
        df.groupby("Año del hecho")
        .size()
        .reset_index(name="total")
        .sort_values("Año del hecho")
    )
    return resultado

res_anio = homicidios_por_anio(df)
res_anio

,Año del hecho,total
0,2015,11585
1,2016,11532
2,2017,11373
3,2018,12130
4,2019,11880
5,2020,11326
6,2021,13238
7,2022,13939
8,2023,14260
9,2024,14021


#### interpretación
- 2015–2017: ligera disminución
- 2018: aumento notable
- 2020: baja (posible efecto pandemia)
- 2021–2023: crecimiento fuerte
- 2024: leve descenso, pero sigue alto

## Top departamentos 

In [13]:
def top_departamentos(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    resultado = (
        df.dropna(subset= ["Departamento del hecho DANE"])
        .groupby("Departamento del hecho DANE")
        .size()
        .reset_index(name="total")
        .sort_values("total", ascending=False)
        .head(top_n)
    )
    return resultado

res_deptos = top_departamentos(df)
res_deptos

,Departamento del hecho DANE,total
32,valle del cauca,23491
1,antioquia,18852
6,"bogotá, d.c.",8992
4,atlántico,6608
12,cauca,6258
24,norte de santander,5461
7,bolívar,4930
15,cundinamarca,4186
23,nariño,3660
21,magdalena,3382


#### Interpretación

El análisis de los departamentos con mayor número de homicidios evidencia una fuerte concentración en regiones específicas del país. Valle del Cauca y Antioquia encabezan el ranking, lo que sugiere violencia asociadas a factores como urbanización, economías ilegales y densidad poblacional.

## Distribución por sexo

In [14]:
def distribucion_sexo(df: pd.DataFrame):
    resultado =  (
        df.groupby("Sexo de la victima")
        .size()
        .reset_index(name="total")
        .sort_values("total", ascending= False)
    )

    resultado["porcentaje"] = (resultado["total"] / resultado["total"].sum() * 100).round(2)
    return resultado 

res_sexo = distribucion_sexo(df)
res_sexo

,Sexo de la victima,total,porcentaje
0,hombre,115147,91.91
2,mujer,10030,8.01
1,indeterminado,107,0.09


#### Interpretación 
La distribución de homicidios por sexo evidencia una marcada predominancia de víctimas masculinas, que representan aproximadamente el 92% del total de casos analizados. Este comportamiento explica que ellos están más expuestos a dinámicas de violencia letal asociadas a contextos urbanos y criminales.

No obstante, el 8% correspondiente a víctimas femeninas reviste especial importancia, ya que puede estar asociado a fenómenos específicos como el feminicidio y la violencia de género.

## Mecanismo causal

In [15]:
def mecanismo_causal(df: pd.DataFrame) -> pd.DataFrame:
    unificaciones = {
        "corto punzante": "cortopunzante",
        "corto contundente": "cortocontundente",
    }

    df_temp = df.copy()
    df_temp["mecanismo"] = (
        df_temp["Mecanismo Causal de la Lesión Fatal"]
        .replace(unificaciones)
    )

    resultado = (
        df_temp.dropna(subset=["mecanismo"])
        .groupby("mecanismo")
        .size()
        .reset_index(name="total")
        .sort_values("total", ascending=False)
    )
    return resultado

res_mecanismo = mecanismo_causal(df)
res_mecanismo

,mecanismo,total
15,proyectil de arma de fuego,92698
6,cortopunzante,21298
3,contundente,3722
5,cortocontundente,2655
9,generadores de asfixia,2569
1,agentes y mecanismo explosivo,631
4,cortante,534
16,punzante,280
14,por determinar,273
19,tóxico,171


#### Interpretación 
El análisis del mecanismo causal de los homicidios evidencia una marcada predominancia del uso de armas de fuego, que representan la mayoría de los casos registrados. Esta tendencia sugiere una alta disponibilidad y uso de este tipo de armas en contextos de violencia letal, lo que incrementa significativamente la probabilidad de muerte en comparación con otros mecanismos.

## Heatmap

In [16]:
def heatmap_depto_circunstancia(df: pd.DataFrame, top_deptos=10, top_circ=8) -> pd.DataFrame:
    df_temp = df.dropna(subset=[
        "Departamento del hecho DANE",
        "Circunstancia del Hecho Detallada"
    ])

    top_d = df_temp.groupby("Departamento del hecho DANE").size().nlargest(top_deptos).index
    top_c = df_temp.groupby("Circunstancia del Hecho Detallada").size().nlargest(top_circ).index

    df_filtrado = df_temp[
        df_temp["Departamento del hecho DANE"].isin(top_d) &
        df_temp["Circunstancia del Hecho Detallada"].isin(top_c)
    ]

    pivot = (
        df_filtrado
        .groupby(["Departamento del hecho DANE", "Circunstancia del Hecho Detallada"])
        .size()
        .reset_index(name="total")
        .pivot(index="Departamento del hecho DANE",
               columns="Circunstancia del Hecho Detallada",
               values="total")
        .fillna(0)
        .astype(int)
    )

    return pivot

res_heatmap = heatmap_depto_circunstancia(df)
res_heatmap

Circunstancia del Hecho Detallada,acción grupos alzados al margen de la ley,ajuste de cuentas,atraco callejero o intento de,otra,otro,riña,sicariato,venganza o ajuste de cuentas
Departamento del hecho DANE,,,,,,,,
antioquia,126,1461,189,369,395,967,888,742
atlántico,1,1181,316,552,529,821,420,115
"bogotá, d.c.",2,225,150,156,223,903,384,0
bolívar,28,484,152,116,71,640,513,52
cauca,126,320,41,48,31,198,391,84
cesar,36,381,147,94,58,240,202,18
magdalena,5,380,44,87,41,138,469,64
nariño,100,75,61,105,190,253,204,80
santander,4,664,89,47,31,792,112,90


## Homicidios por zona y año

In [17]:
def homicidios_por_zona_anio(df: pd.DataFrame) -> pd.DataFrame:
    resultado = (
        df.dropna(subset=["Zona del Hecho"])
        .groupby(["Zona del Hecho", "Año del hecho"])
        .size()
        .reset_index(name="total")
        .sort_values(["Zona del Hecho", "Año del hecho"])
    )
    return resultado

res_zona = homicidios_por_zona_anio(df)
res_zona

,Zona del Hecho,Año del hecho,total
0,cabecera municipal,2015,8842
1,cabecera municipal,2016,8731
2,cabecera municipal,2017,8228
3,cabecera municipal,2018,8421
4,cabecera municipal,2019,8099
5,cabecera municipal,2020,7402
6,cabecera municipal,2021,8860
7,cabecera municipal,2022,9762
8,cabecera municipal,2023,9935
9,cabecera municipal,2024,9828


#### Interpretación 

El análisis de los homicidios según la zona del hecho evidencia una clara predominancia de los casos en cabeceras municipales, lo que confirma el carácter mayoritariamente urbano de la violencia homicida. No obstante, las zonas rurales presentan una dinámica relevante, con un crecimiento sostenido en los últimos años, lo que sugiere la persistencia de factores estructurales asociados a conflictos territoriales y economías ilegales. Por su parte, los centros poblados registran una menor incidencia, aunque muestran una tendencia al aumento. Estas diferencias reflejan la necesidad de enfoques diferenciados en las políticas de seguridad según el contexto territorial.


## Feminicidios desde 2018

In [18]:
def feminicidios_desde_2018(df: pd.DataFrame) -> pd.DataFrame:
    df_femi = df[
        (df["Año del hecho"] >= 2018) &
        (df["Circunstancia del Hecho Detallada"]
         .str.contains("femini", na=False, case=False))
    ]

    resultado = (
        df_femi.groupby("Año del hecho")
        .size()
        .reset_index(name="total_feminicidios")
        .sort_values("Año del hecho")
    )

    return resultado

res_femi = feminicidios_desde_2018(df)
res_femi

,Año del hecho,total_feminicidios
0,2018,78
1,2019,109
2,2020,90
3,2021,59
4,2022,110
5,2023,204
6,2024,158


#### Interpretacióm 
El análisis de feminicidios desde 2018 evidencia una tendencia creciente en el número de casos, con un incremento particularmente significativo en 2023. Aunque se observan fluctuaciones en años como 2020 y 2021, posiblemente asociadas a dinámicas de confinamiento o subregistro, la tendencia general sugiere un aumento preocupante de la violencia basada en género. Estos resultados resaltan la necesidad de fortalecer las políticas públicas orientadas a la prevención, atención y sanción del feminicidio.

In [20]:
def ejecutar_analisis(df: pd.DataFrame) -> dict:
    """
    Ejecuta todas las funciones de análisis y retorna un diccionario
    con cada DataFrame resultado.
    """
    print("\n====== PASO 5: ANÁLISIS AGREGADO ======\n")

    resultados = {
        "por_anio":     homicidios_por_anio(df),
        "top_deptos":   top_departamentos(df),
        "por_sexo":     distribucion_sexo(df),
        "mecanismo":    mecanismo_causal(df),
        "heatmap":      heatmap_depto_circunstancia(df),
        "zona_anio":    homicidios_por_zona_anio(df),
        "feminicidios": feminicidios_desde_2018(df),
    }

    print("\n====== ANÁLISIS COMPLETADO ======\n")
    return resultados